# Test Set Evaluation

Final evaluation on actors 21-24 (2M + 2F). This set was never used during training or hyperparameter tuning across any of the three models. Results here are the honest generalization numbers.

All three architectures are evaluated side by side to compare what each approach learned.

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import glob
import os
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from data_loader import prepare_data

## Config

In [ ]:
DATASET_PATH = "Audio_Speech_Actors_01-24"
MAX_LEN      = 300
BATCH_SIZE   = 32

EMOTION_LABELS = [
    'neutral', 'calm', 'happy', 'sad',
    'angry', 'fearful', 'disgust', 'surprised'
]

feature_files = glob.glob(os.path.join(DATASET_PATH, "Actor_*", "*.npz"))

## Model Definitions

Each architecture is defined here exactly as it was during training. Weights are loaded from the saved checkpoints.

In [ ]:
class MLP(nn.Module):
    def __init__(self, dropout_coeff=0.3):
        super().__init__()
        self.fc1     = nn.Linear(122, 256)
        self.fc2     = nn.Linear(256, 128)
        self.fc3     = nn.Linear(128, 64)
        self.fc4     = nn.Linear(64, 8)
        self.dropout = nn.Dropout(dropout_coeff)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        return x


class CNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
        self.conv1   = nn.Conv1d(in_channels=41, out_channels=32, kernel_size=5)
        self.conv2   = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5)
        self.conv3   = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5)
        self.pool    = nn.MaxPool1d(2, 2)
        self.bn1     = nn.BatchNorm1d(32)
        self.bn2     = nn.BatchNorm1d(64)
        self.bn3     = nn.BatchNorm1d(128)
        self.fc1     = nn.Linear(128, 64)
        self.fc2     = nn.Linear(64, 8)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(self.pool(F.relu(self.bn1(self.conv1(x)))))
        x = self.dropout(self.pool(F.relu(self.bn2(self.conv2(x)))))
        x = self.dropout(self.pool(F.relu(self.bn3(self.conv3(x)))))
        x = x.mean(dim=-1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class GRU(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.gru     = nn.GRU(input_size=41, hidden_size=64, num_layers=2, batch_first=True)
        self.fc1     = nn.Linear(64, 32)
        self.fc2     = nn.Linear(32, 8)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        output, _ = self.gru(x)
        x = output[:, -1, :]
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

## Load Test Data

Each model requires a different input format so three separate data loading calls are made.

- MLP: flat 122-dim vector, all features averaged over time
- CNN: `(41, 300)` — MFCC + RMS, time preserved
- GRU: `(300, 41)` — same as CNN but transposed

In [ ]:
# MLP data
[_, _, _, _, x_test_mlp, y_test] = prepare_data(
    feature_files=feature_files,
    flatten=True,
    features=['mfcc', 'delta', 'delta2', 'zcr', 'rms'],
    max_len=MAX_LEN
)

# CNN data
[_, _, _, _, x_test_cnn, _] = prepare_data(
    feature_files=feature_files,
    flatten=False,
    features=['mfcc', 'rms'],
    max_len=MAX_LEN
)

# GRU data — same as CNN but transposed
x_test_gru = x_test_cnn.permute(0, 2, 1)

y_test_torch = y_test

print(f"Test samples: {len(y_test_torch)}")
print(f"Label distribution: {torch.bincount(y_test_torch)}")

## Load Weights

In [ ]:
mlp = MLP()
mlp.load_state_dict(torch.load('mlp_weights.pth'))
mlp.eval()

cnn = CNN()
cnn.load_state_dict(torch.load('cnn_best.pth'))
cnn.eval()

gru = GRU()
gru.load_state_dict(torch.load('gru_best.pth'))
gru.eval()

print("All models loaded")

## Evaluation Helper

In [ ]:
def evaluate(model, x, y, name):
    loader = DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=False)
    all_preds  = []
    all_labels = []
    correct = 0
    total   = 0

    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.numpy())
            all_labels.extend(labels.numpy())
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"\n{'='*50}")
    print(f"{name} — Test Accuracy: {accuracy:.2f}%")
    print('='*50)
    print(classification_report(all_labels, all_preds, target_names=EMOTION_LABELS))

    return all_labels, all_preds, accuracy

## Results

In [ ]:
labels_mlp, preds_mlp, acc_mlp = evaluate(mlp, x_test_mlp, y_test_torch, 'MLP')
labels_cnn, preds_cnn, acc_cnn = evaluate(cnn, x_test_cnn, y_test_torch, 'CNN')
labels_gru, preds_gru, acc_gru = evaluate(gru, x_test_gru, y_test_torch, 'GRU')

## Accuracy Comparison

In [ ]:
models      = ['MLP', 'CNN', 'GRU']
accuracies  = [acc_mlp, acc_cnn, acc_gru]
random_chance = 100 / 8  # 8 classes

plt.figure(figsize=(6, 4))
plt.bar(models, accuracies, color=['steelblue', 'darkorange', 'seagreen'])
plt.axhline(random_chance, color='red', linestyle='--', label=f'Random chance ({random_chance:.1f}%)')
plt.ylabel('Test Accuracy (%)')
plt.title('Test Set Accuracy by Model')
plt.legend()
plt.tight_layout()
plt.show()

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, labels, preds, name in zip(
    axes,
    [labels_mlp, labels_cnn, labels_gru],
    [preds_mlp,  preds_cnn,  preds_gru],
    ['MLP', 'CNN', 'GRU']
):
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=EMOTION_LABELS)
    disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(name)

plt.tight_layout()
plt.show()